# LeetCode 2222 — Number of Ways to Select Buildings bằng Python

Link đề: https://leetcode.com/problems/number-of-ways-to-select-buildings/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def numberOfWays(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu "không có hai tòa **liên tiếp** (trong 3 tòa được chọn) cùng loại" trên chuỗi nhị phân ⇔ dãy con được chọn là `"010"` hoặc `"101"`.
2. Đếm bằng kỹ thuật **cố định phần tử giữa**: số cách = (số phần tử khác loại bên trái) × (số phần tử khác loại bên phải).
3. Đếm bằng **DP đếm dãy con** theo mẫu: giữ số dãy con `"0"`, `"01"`, `"010"` (và tương tự cho `"101"`) trong một lượt.
4. Tổng quát hóa sang chọn `k` tòa xen kẽ.
5. Kiểm thử đối chiếu brute force `O(n³)` trên mọi chuỗi ngắn và dữ liệu ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Chuỗi nhị phân `s`: `'0'` là văn phòng, `'1'` là nhà hàng. Chọn **3 tòa** (chỉ số `i < j < k`) sao cho **không có hai tòa liền nhau trong bộ được chọn cùng loại** — tức `s[i] ≠ s[j]` và `s[j] ≠ s[k]`. Đếm số cách chọn.

**Ví dụ 1:** `s = "001101"` → `6`: `[0,2,4]`, `[0,3,4]`, `[1,2,4]`, `[1,3,4]` (cùng tạo `"010"`), `[2,4,5]`, `[3,4,5]` (tạo `"101"`).

**Ví dụ 2:** `s = "11100"` → `0`.

**Ràng buộc:** `3 ≤ n ≤ 10⁵`.

In [1]:
import random
import time
from itertools import combinations, product
from typing import List

examples = [("001101", 6), ("11100", 0)]
for s, exp in examples:
    print(repr(s), "-> kỳ vọng", exp)

'001101' -> kỳ vọng 6
'11100' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Chuỗi nhị phân độ dài 3 "xen kẽ"

Với 3 ký tự nhị phân, `a ≠ b` và `b ≠ c` buộc `a = c` ⇒ chỉ có `"010"` và `"101"`.

In [2]:
valid = ["".join(t) for t in product("01", repeat=3) if t[0] != t[1] and t[1] != t[2]]
print(valid)

['010', '101']


### 3.2 Đếm theo phần tử giữa — quy tắc nhân

Cố định `j` là tòa ở giữa với `s[j] = c`. Tòa đầu phải ở bên trái, khác loại `c`; tòa cuối ở bên phải, khác loại `c`. Hai lựa chọn độc lập ⇒ số cách = `left_khác(j) × right_khác(j)`.

In [3]:
s, j = "001101", 4                       # s[4] = '0'
left = s[:j].count("1"); right = s[j + 1:].count("1")
print(f"giữa tại j={j} ('{s[j]}'): trái có {left} số '1', phải có {right} số '1' -> {left * right} cách")

giữa tại j=4 ('0'): trái có 2 số '1', phải có 1 số '1' -> 2 cách


### 3.3 Đếm dãy con theo mẫu bằng DP (một lượt)

Đếm số dãy con bằng mẫu `p = p₀p₁p₂`: giữ `cnt[t]` = số dãy con bằng `p[:t]` trong tiền tố đã duyệt. Gặp ký tự `ch`: với `t` **giảm dần** từ 3 về 1, nếu `p[t−1] == ch` thì `cnt[t] += cnt[t−1]` (với `cnt[0] = 1`). Giảm dần để không dùng `ch` hai lần.

In [4]:
def count_pattern(s, p):
    cnt = [1] + [0] * len(p)
    for ch in s:
        for t in range(len(p), 0, -1):
            if p[t - 1] == ch:
                cnt[t] += cnt[t - 1]
    return cnt[-1]

print(count_pattern("001101", "010"), count_pattern("001101", "101"))

4 2


### 3.4 Tổng tiền tố số lượng

`ones_before[j]` = số `'1'` trong `s[:j]`; số `'1'` bên phải `j` = `total_ones − ones_before[j] − (s[j] == '1')`.

In [5]:
s = "001101"
ones_before = [0]
for ch in s:
    ones_before.append(ones_before[-1] + (ch == "1"))
print("ones_before =", ones_before, "| tổng '1' =", ones_before[-1])

ones_before = [0, 0, 0, 1, 2, 2, 3] | tổng '1' = 3


### 3.5 Dãy con ≠ xâu con liên tiếp

Ba tòa được chọn **không cần liền nhau** trên phố; "liên tiếp" trong đề là liên tiếp **trong bộ được chọn**. Vì vậy ta đếm **dãy con** `"010"`/`"101"`, không phải xâu con. `str.count` chỉ đếm xâu con liên tiếp (và không chồng nhau) — sai hoàn toàn cho bài này.

In [6]:
s = "001101"
print("xâu con '010' (str.count):", s.count("010"), "| dãy con '010':", count_pattern(s, "010"))
print("xâu con '101':", s.count("101"), "| dãy con '101':", count_pattern(s, "101"))

xâu con '010' (str.count): 0 | dãy con '010': 4
xâu con '101': 1 | dãy con '101': 2


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `s = "001101"`. Cố định từng vị trí làm tòa giữa:

| j | s[j] | khác loại bên trái | khác loại bên phải | số cách |
|---|---|---|---|---|
| 0 | 0 | 0 | 3 | 0 |
| 1 | 0 | 0 | 3 | 0 |
| 2 | 1 | 2 | 1 | 2 |
| 3 | 1 | 2 | 1 | 2 |
| 4 | 0 | 2 | 1 | 2 |
| 5 | 1 | 3 | 0 | 0 |

Tổng `6` ✓. Mỗi bộ `(i, j, k)` hợp lệ được đếm **đúng một lần** — tại tòa giữa `j` của nó.

In [7]:
s = "001101"; total = 0
for j, c in enumerate(s):
    other = "1" if c == "0" else "0"
    L = s[:j].count(other); R = s[j + 1:].count(other); total += L * R
    print(f"j={j} '{c}': trái {L}, phải {R} -> {L * R} (cộng dồn {total})")

j=0 '0': trái 0, phải 3 -> 0 (cộng dồn 0)
j=1 '0': trái 0, phải 3 -> 0 (cộng dồn 0)
j=2 '1': trái 2, phải 1 -> 2 (cộng dồn 2)
j=3 '1': trái 2, phải 1 -> 2 (cộng dồn 4)
j=4 '0': trái 2, phải 1 -> 2 (cộng dồn 6)
j=5 '1': trái 3, phải 0 -> 0 (cộng dồn 6)


Để "thấy" 6 cách chọn của ví dụ 1, in chuỗi với ba tòa được chọn viết hoa trong ngoặc. Để ý các cách được gom theo **tòa giữa**: hai cách có giữa tại 2, hai cách giữa tại 3, hai cách giữa tại 4 — đúng như bảng trên.

In [8]:
s = "001101"
for i, j, k in combinations(range(len(s)), 3):
    if s[i] != s[j] and s[j] != s[k]:
        mark = "".join(f"[{c}]" if t in (i, j, k) else f" {c} " for t, c in enumerate(s))
        print(f"({i},{j},{k}) giữa={j}: {mark} -> {s[i]}{s[j]}{s[k]}")

(0,2,4) giữa=2: [0] 0 [1] 1 [0] 1  -> 010
(0,3,4) giữa=3: [0] 0  1 [1][0] 1  -> 010
(1,2,4) giữa=2:  0 [0][1] 1 [0] 1  -> 010
(1,3,4) giữa=3:  0 [0] 1 [1][0] 1  -> 010
(2,4,5) giữa=4:  0  0 [1] 1 [0][1] -> 101
(3,4,5) giữa=4:  0  0  1 [1][0][1] -> 101


## 5. Cách 1 — Brute Force / baseline: mọi bộ ba chỉ số

**Ý tưởng:** duyệt `combinations(range(n), 3)`, kiểm `s[i] != s[j] != s[k]` (theo nghĩa hai cặp liền nhau khác nhau).

In [9]:
def ways_brute(s):
    return sum(1 for i, j, k in combinations(range(len(s)), 3) if s[i] != s[j] and s[j] != s[k])

for s, exp in examples:
    got = ways_brute(s); print(repr(s), "->", got); assert got == exp

'001101' -> 6
'11100' -> 0


**Độ phức tạp:** `C(n, 3) ≈ n³/6` — với `n = 10⁵` là `1,7·10¹⁴`. Không thể. Space `O(1)`.

**Nút thắt:** khi cố định `j`, việc chọn `i` và `k` **độc lập** — chỉ cần **đếm**, không cần liệt kê.

In [10]:
rng = random.Random(0)
for n in (50, 100, 200):
    s = "".join(rng.choice("01") for _ in range(n))
    t = time.perf_counter(); ways_brute(s)
    print(f"n={n}: {time.perf_counter() - t:.3f}s (gấp đôi n -> ~x8)")

n=50: 0.001s (gấp đôi n -> ~x8)
n=100: 0.007s (gấp đôi n -> ~x8)
n=200: 0.054s (gấp đôi n -> ~x8)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (phần tử giữa):** đáp án `= Σ_j L(j) · R(j)`, với `L(j)`, `R(j)` = số tòa khác loại `s[j]` ở bên trái/phải `j`. Duyệt một lượt, giữ số `'0'` và `'1'` đã gặp; số bên phải = tổng − đã gặp − (chính nó).

**Insight 2 (DP mẫu):** đáp án `= #("010") + #("101")` (số dãy con). Đếm cả hai mẫu cùng lúc bằng 6 bộ đếm: `c0, c1` (dãy con `"0"`, `"1"`), `c01, c10`, `c010, c101`.
- Gặp `'0'`: `c010 += c01`; `c10 += c1`; `c0 += 1`.
- Gặp `'1'`: `c101 += c10`; `c01 += c0`; `c1 += 1`.

**Quy tắc "nếu… thì…":** **nếu** `s[j] = '0'` **thì** nó chỉ có thể là giữa của `"101"` ⇒ cộng `(#1 trái)·(#1 phải)`; **nếu** `'1'` thì giữa của `"010"` ⇒ `(#0 trái)·(#0 phải)`.

**Invariant (DP):** sau khi đọc tiền tố `s[:t]`, mỗi bộ đếm `c_p` = số dãy con của tiền tố bằng mẫu `p`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Phần tử giữa với mảng tiền tố

In [11]:
def ways_prefix(s):
    n = len(s)
    ones = [0] * (n + 1)
    for i, ch in enumerate(s):
        ones[i + 1] = ones[i] + (ch == "1")
    total_ones = ones[n]
    ans = 0
    for j, ch in enumerate(s):
        ones_left = ones[j]; zeros_left = j - ones_left
        ones_right = total_ones - ones[j + 1]; zeros_right = (n - j - 1) - ones_right
        ans += zeros_left * zeros_right if ch == "1" else ones_left * ones_right
    return ans

for s, exp in examples:
    assert ways_prefix(s) == exp
print([ways_prefix(s) for s, _ in examples])

[6, 0]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Phần tử giữa, bộ đếm chạy `O(1)` bộ nhớ (bản nộp chính)

In [12]:
def ways_running(s):
    total0 = s.count("0"); total1 = len(s) - total0
    seen0 = seen1 = 0; ans = 0
    for ch in s:
        if ch == "0":
            ans += seen1 * (total1 - seen1)          # giữa của "101"
            seen0 += 1
        else:
            ans += seen0 * (total0 - seen0)          # giữa của "010"
            seen1 += 1
    return ans

for s, exp in examples:
    assert ways_running(s) == exp
print([ways_running(s) for s, _ in examples])

[6, 0]


Lưu ý: khi `ch == '0'`, số `'1'` bên phải là `total1 − seen1` (chính `ch` không phải `'1'` nên không cần trừ thêm).

**Độ phức tạp Cách 3:** `O(n)`, `O(1)` bộ nhớ.

### 7.3 Cách 4 — DP đếm dãy con `"010"` và `"101"` cùng lúc

In [13]:
def ways_dp(s):
    c0 = c1 = c01 = c10 = c010 = c101 = 0
    for ch in s:
        if ch == "0":
            c010 += c01; c10 += c1; c0 += 1
        else:
            c101 += c10; c01 += c0; c1 += 1
    return c010 + c101

for s, exp in examples:
    assert ways_dp(s) == exp
print([ways_dp(s) for s, _ in examples])

[6, 0]


In [14]:
class SolutionDP:
    def numberOfWays(self, s: str) -> int:
        c0 = c1 = c01 = c10 = c010 = c101 = 0
        for ch in s:
            if ch == "0":
                c010 += c01; c10 += c1; c0 += 1
            else:
                c101 += c10; c01 += c0; c1 += 1
        return c010 + c101

print([SolutionDP().numberOfWays(s) for s, _ in examples])

[6, 0]


**Độ phức tạp Cách 4:** `O(n)`, `O(1)`. Ưu điểm: tổng quát hóa dễ cho mẫu dài hơn hoặc chọn `k` tòa (bài tập 5).

### 7.4 Cách 5 — Hàm đếm mẫu tổng quát `count_pattern`

Dùng hàm ở mục 3.3 cho hai mẫu. Chậm hơn hằng số nhưng tái sử dụng được cho mọi mẫu.

In [15]:
def ways_generic(s):
    return count_pattern(s, "010") + count_pattern(s, "101")

for s, exp in examples:
    assert ways_generic(s) == exp
print([ways_generic(s) for s, _ in examples])

[6, 0]


## 8. TRACE từng bước

Trace Cách 4 trên ví dụ 1 `"001101"`:

| ch | c0 | c1 | c01 | c10 | c010 | c101 |
|---|---|---|---|---|---|---|
| 0 | 1 | 0 | 0 | 0 | 0 | 0 |
| 0 | 2 | 0 | 0 | 0 | 0 | 0 |
| 1 | 2 | 1 | 2 | 0 | 0 | 0 |
| 1 | 2 | 2 | 4 | 0 | 0 | 0 |
| 0 | 3 | 2 | 4 | 2 | 4 | 0 |
| 1 | 3 | 3 | 7 | 2 | 4 | 2 |

Đáp án `4 + 2 = 6` ✓.

In [16]:
def trace_dp(s):
    c0 = c1 = c01 = c10 = c010 = c101 = 0
    print(" ch | c0 c1 | c01 c10 | c010 c101")
    for ch in s:
        if ch == "0": c010 += c01; c10 += c1; c0 += 1
        else: c101 += c10; c01 += c0; c1 += 1
        print(f"  {ch} | {c0:2d} {c1:2d} | {c01:3d} {c10:3d} | {c010:4d} {c101:4d}")
    return c010 + c101

print("Đáp án:", trace_dp("001101"))

 ch | c0 c1 | c01 c10 | c010 c101
  0 |  1  0 |   0   0 |    0    0
  0 |  2  0 |   0   0 |    0    0
  1 |  2  1 |   2   0 |    0    0
  1 |  2  2 |   4   0 |    0    0
  0 |  3  2 |   4   2 |    4    0
  1 |  3  3 |   7   2 |    4    2
Đáp án: 6


In [17]:
def trace_running(s):
    total0 = s.count("0"); total1 = len(s) - total0; seen0 = seen1 = 0; ans = 0
    for j, ch in enumerate(s):
        if ch == "0":
            add = seen1 * (total1 - seen1); kind = f"'101': {seen1} x {total1 - seen1}"; seen0 += 1
        else:
            add = seen0 * (total0 - seen0); kind = f"'010': {seen0} x {total0 - seen0}"; seen1 += 1
        ans += add
        print(f"j={j} '{ch}' giữa của {kind} = {add} -> ans={ans}")
    return ans

print("Đáp án:", trace_running("0101100"))

j=0 '0' giữa của '101': 0 x 3 = 0 -> ans=0
j=1 '1' giữa của '010': 1 x 3 = 3 -> ans=3
j=2 '0' giữa của '101': 1 x 2 = 2 -> ans=5
j=3 '1' giữa của '010': 2 x 2 = 4 -> ans=9
j=4 '1' giữa của '010': 2 x 2 = 4 -> ans=13
j=5 '0' giữa của '101': 3 x 0 = 0 -> ans=13
j=6 '0' giữa của '101': 3 x 0 = 0 -> ans=13
Đáp án: 13


## 9. Code nộp LeetCode

In [18]:
class Solution:
    def numberOfWays(self, s: str) -> int:
        total0 = s.count("0")
        total1 = len(s) - total0
        seen0 = seen1 = 0
        ans = 0
        for ch in s:                                   # ch là tòa ở GIỮA
            if ch == "0":
                ans += seen1 * (total1 - seen1)        # "1 0 1"
                seen0 += 1
            else:
                ans += seen0 * (total0 - seen0)        # "0 1 0"
                seen1 += 1
        return ans

for s, exp in examples:
    assert Solution().numberOfWays(s) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Phân hoạch theo phần tử giữa.** Mỗi bộ hợp lệ `(i < j < k)` có đúng một phần tử giữa `j`. Với `j` cố định và `c = s[j]`, bộ hợp lệ ⇔ `s[i] ≠ c` và `s[k] ≠ c` (với `i < j < k`). Số `i` thỏa = số ký tự khác `c` trong `s[:j]`; số `k` = số ký tự khác `c` trong `s[j+1:]`; hai lựa chọn độc lập ⇒ tích. Tổng theo `j` đếm mỗi bộ đúng một lần.

**Bộ đếm chạy (Cách 3).** Khi xử lý `j` với `ch = '0'`: `seen1` = số `'1'` trong `s[:j]`, `total1 − seen1` = số `'1'` trong `s[j+1:]` (vì `s[j]` không phải `'1'`). Tương tự cho `'1'`.

**DP (Cách 4).** Bất biến: sau tiền tố `s[:t]`, `c_p` = số dãy con bằng `p`. Khi thêm `ch`: các dãy con **mới** bằng `p` là các dãy con kết thúc tại `ch`, tức một dãy con bằng `p[:-1]` của tiền tố cũ nối thêm `ch` (nếu `p[-1] == ch`). Cập nhật `c010 += c01` **trước** khi các bộ đếm bậc thấp thay đổi do chính `ch` (thứ tự bậc cao → thấp), nên không dùng `ch` hai lần.

Kiểm chứng phân hoạch theo phần tử giữa **từng `j` một**: số bộ hợp lệ có giữa tại `j` (đếm thẳng) phải bằng `L(j)·R(j)`.

In [19]:
rng = random.Random(10)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(3, 12)))
    for j in range(len(s)):
        direct = sum(1 for i in range(j) for k in range(j + 1, len(s)) if s[i] != s[j] and s[k] != s[j])
        other = "1" if s[j] == "0" else "0"
        assert direct == s[:j].count(other) * s[j + 1:].count(other)
print("Số bộ có giữa tại j == L(j)·R(j) với mọi j trên 300 chuỗi")

Số bộ có giữa tại j == L(j)·R(j) với mọi j trên 300 chuỗi


## 11. Phân tích Time / Space

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi bộ ba | `O(n³)` | `O(1)` |
| 2. Giữa + tiền tố | `O(n)` | `O(n)` |
| 3. Giữa + bộ đếm chạy | `O(n)` | `O(1)` |
| 4. DP 6 bộ đếm | `O(n)` | `O(1)` |
| 5. `count_pattern` | `O(n · |p|)` | `O(|p|)` |

Đáp án tối đa cỡ `n³/27 ≈ 3,7·10¹³` với `n = 10⁵` — cần `long` trong Java/C++.

In [20]:
rng = random.Random(2222)
big = "".join(rng.choice("01") for _ in range(10**5))
alt = "01" * (5 * 10**4)
for name, fn in [("prefix", ways_prefix), ("running", ways_running), ("dp", ways_dp), ("generic", ways_generic),
                 ("Solution", lambda s: Solution().numberOfWays(s))]:
    t = time.perf_counter(); r1 = fn(big); r2 = fn(alt)
    print(f"{name:9s}: ngẫu nhiên={r1}, xen kẽ={r2} ({time.perf_counter() - t:.3f}s)")

prefix   : ngẫu nhiên=41662169452446, xen kẽ=41666666650000 (0.034s)


running  : ngẫu nhiên=41662169452446, xen kẽ=41666666650000 (0.012s)


dp       : ngẫu nhiên=41662169452446, xen kẽ=41666666650000 (0.011s)


generic  : ngẫu nhiên=41662169452446, xen kẽ=41666666650000 (0.059s)
Solution : ngẫu nhiên=41662169452446, xen kẽ=41666666650000 (0.011s)


## 12. Bộ test

- Ví dụ đề.
- Biên: `n = 3` (`"010"` → 1, `"011"` → 0); chuỗi toàn một ký tự (0); xen kẽ hoàn toàn.
- **Vét cạn** mọi chuỗi nhị phân độ dài `3..12` so với brute force; ngẫu nhiên lớn so các cách với nhau (seed cố định).

In [21]:
fns = {"brute": ways_brute, "prefix": ways_prefix, "running": ways_running, "dp": ways_dp,
       "generic": ways_generic, "Solution": lambda s: Solution().numberOfWays(s), "SolutionDP": lambda s: SolutionDP().numberOfWays(s)}
edge = [("010", 1), ("101", 1), ("011", 0), ("000", 0), ("1111111", 0), ("0101", 2), ("01010", ways_brute("01010"))]
for s, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(s) == exp, (name, s, fn(s))
cnt = 0
for n in range(3, 13):
    for tup in product("01", repeat=n):
        s = "".join(tup); want = ways_brute(s)
        for name, fn in fns.items():
            if name != "brute": assert fn(s) == want, (name, s)
        cnt += 1
print(f"Ví dụ + biên + vét cạn {cnt} chuỗi: OK")

Ví dụ + biên + vét cạn 8184 chuỗi: OK


In [22]:
rng = random.Random(42)
for _ in range(200):
    p = rng.random()
    s = "".join("1" if rng.random() < p else "0" for _ in range(rng.randint(3, 2000)))
    r = ways_dp(s)
    assert ways_prefix(s) == ways_running(s) == ways_generic(s) == Solution().numberOfWays(s) == r
print("200 test ngẫu nhiên lớn: các cách khớp nhau")

200 test ngẫu nhiên lớn: các cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm `"010"` (quên `"101"`)

In [23]:
def wrong_only_010(s):
    return count_pattern(s, "010")                         # SAI: thiếu "101"

print("sai:", wrong_only_010("001101"), "| đúng:", 6)

sai: 4 | đúng: 6


In [24]:
print("sửa:", ways_generic("001101"))

sửa: 6


### Lỗi 2 — Nhân số **cùng loại** thay vì khác loại ở hai bên

In [25]:
def wrong_same_type(s):
    ans = 0
    for j, ch in enumerate(s):
        ans += s[:j].count(ch) * s[j + 1:].count(ch)        # SAI: phải là loại khác
    return ans

print("sai:", wrong_same_type("001101"), "| đúng:", 6)

sai: 2 | đúng: 6


In [26]:
print("sửa:", Solution().numberOfWays("001101"))

sửa: 6


### Lỗi 3 — Hiểu nhầm thành "ba tòa đôi một khác loại"

Với chuỗi nhị phân, ba tòa không thể đôi một khác loại ⇒ đáp án luôn 0. Đề chỉ yêu cầu hai tòa **liền nhau trong bộ chọn** khác loại (tòa 1 và tòa 3 được phép cùng loại).

In [27]:
def wrong_pairwise(s):
    return sum(1 for i, j, k in combinations(range(len(s)), 3) if len({s[i], s[j], s[k]}) == 3)   # SAI

print("sai:", wrong_pairwise("001101"), "| đúng:", 6)

sai: 0 | đúng: 6


In [28]:
print("sửa:", ways_brute("001101"))

sửa: 6


### Lỗi 4 — Bộ đếm bên phải tính sai (trừ nhầm chính phần tử giữa)

Khi `ch = '0'`, số `'1'` bên phải là `total1 − seen1`; nếu "cẩn thận thừa" trừ thêm 1 thì sai.

In [29]:
def wrong_right_count(s):
    total0 = s.count("0"); total1 = len(s) - total0; seen0 = seen1 = 0; ans = 0
    for ch in s:
        if ch == "0":
            ans += seen1 * (total1 - seen1 - 1); seen0 += 1     # SAI: -1 thừa
        else:
            ans += seen0 * (total0 - seen0 - 1); seen1 += 1
    return ans

print("sai:", wrong_right_count("001101"), "| đúng:", 6)

sai: -3 | đúng: 6


In [30]:
print("sửa:", ways_running("001101"))

sửa: 6


### Lỗi 5 — DP cập nhật bậc thấp trước bậc cao (khi mẫu có ký tự lặp)

Trong `count_pattern`, nếu duyệt `t` **tăng dần**, ký tự hiện tại có thể được dùng hai lần trong cùng một dãy con. Lỗi chỉ lộ ra khi mẫu có hai ký tự **giống nhau đứng liền nhau** (như `"00"`); với `"010"` (hai chữ `0` không liền nhau) nó tình cờ vẫn đúng — một lý do nữa để luôn test nhiều mẫu.

In [31]:
def wrong_count_pattern(s, p):
    cnt = [1] + [0] * len(p)
    for ch in s:
        for t in range(1, len(p) + 1):                        # SAI: tăng dần
            if p[t - 1] == ch:
                cnt[t] += cnt[t - 1]
    return cnt[-1]

print("sai:", wrong_count_pattern("0110", "00") , "| đúng:", count_pattern("0110", "00"),
      "| với '010':", wrong_count_pattern("001101", "010"), "vs", count_pattern("001101", "010"))

sai: 3 | đúng: 1 | với '010': 4 vs 4


In [32]:
print("sửa (giảm dần):", count_pattern("0110", "00"), count_pattern("001101", "010"))

sửa (giảm dần): 1 4


### Lỗi 6 — Đếm bằng tổ hợp bỏ qua **thứ tự vị trí**

Ý nghĩ "chọn hai số 0 và một số 1 rồi xếp thành 010" cho `C(c0, 2)·c1` — sai vì số `1` phải nằm **giữa** hai số `0` theo vị trí, không phải bất kỳ.

In [33]:
from math import comb as _comb
def wrong_ignore_order(s):
    c0, c1 = s.count("0"), s.count("1")
    return _comb(c0, 2) * c1 + _comb(c1, 2) * c0               # SAI: không quan tâm vị trí

print("sai:", wrong_ignore_order("001101"), "| đúng:", 6, "| '11100' sai:", wrong_ignore_order("11100"), "vs 0")

sai: 18 | đúng: 6 | '11100' sai: 9 vs 0


In [34]:
print("sửa:", Solution().numberOfWays("001101"), Solution().numberOfWays("11100"))

sửa: 6 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Đếm bộ ba `i < j < k` thỏa điều kiện cục bộ** ⇒ **cố định phần tử giữa**, nhân số cách bên trái và bên phải.
- **Đếm dãy con bằng một mẫu cố định** ⇒ DP các tiền tố của mẫu, cập nhật **bậc cao trước**.
- Chuỗi nhị phân với "liền kề khác nhau" ⇒ chỉ có vài mẫu (`010`, `101`).
- Đáp án lớn ⇒ `long` ở ngôn ngữ khác.
- Vét cạn mọi chuỗi nhị phân ngắn là cách kiểm thử rất mạnh.

## 15. Biến thể của bài

1. **Chọn `k` tòa xen kẽ** (bài tập 5): DP với `2k` bộ đếm (dãy con xen kẽ độ dài `t` kết thúc bằng `0`/`1`).
2. **LeetCode 1987 — Number of Unique Good Subsequences (Hard):** đếm dãy con **khác nhau**.
3. **LeetCode 115 — Distinct Subsequences (Hard):** mẫu tùy ý, DP như `count_pattern`.
4. **Đếm bộ ba `i<j<k` với `a[i] < a[j] < a[k]`** (LeetCode 2179-kiểu): cố định giữa, đếm bên trái nhỏ hơn và bên phải lớn hơn (cần BIT khi giá trị lớn).

Minh họa biến thể 4 (tự dựng, kiểm bằng brute force):

In [35]:
def increasing_triplets(a):
    n = len(a); ans = 0
    for j in range(n):
        left = sum(1 for i in range(j) if a[i] < a[j])
        right = sum(1 for k in range(j + 1, n) if a[k] > a[j])
        ans += left * right
    return ans

rng = random.Random(4)
for _ in range(300):
    a = [rng.randint(0, 5) for _ in range(rng.randint(3, 10))]
    assert increasing_triplets(a) == sum(1 for i, j, k in combinations(range(len(a)), 3) if a[i] < a[j] < a[k])
print("Đếm bộ ba tăng (cố định phần tử giữa) khớp brute force; [1,2,3,4] ->", increasing_triplets([1, 2, 3, 4]))

Đếm bộ ba tăng (cố định phần tử giữa) khớp brute force; [1,2,3,4] -> 4


Minh họa thêm (tự dựng) — **bảng chữ 3 loại** (`'0'` văn phòng, `'1'` nhà hàng, `'2'` cửa hàng): đếm số cách chọn 3 tòa sao cho hai tòa liền nhau trong bộ chọn khác loại. Lúc này `s[i]` và `s[k]` có thể khác nhau (ví dụ `"012"`), nên công thức phần tử giữa trở thành `(số khác loại bên trái) × (số khác loại bên phải)` với "khác loại" nghĩa là khác `s[j]` — vẫn đúng y nguyên!

In [36]:
def ways_alpha(s):
    total = {c: s.count(c) for c in set(s)}; seen = {c: 0 for c in total}; n = len(s); ans = 0
    for j, ch in enumerate(s):
        left_diff = j - seen[ch]
        right_diff = (n - j - 1) - (total[ch] - seen[ch] - 1)
        ans += left_diff * right_diff
        seen[ch] += 1
    return ans

rng = random.Random(3)
for _ in range(400):
    s = "".join(rng.choice("012") for _ in range(rng.randint(3, 11)))
    assert ways_alpha(s) == ways_brute(s)
print("Bảng chữ 3 loại: công thức phần tử giữa khớp brute force; '0120' ->", ways_alpha("0120"))

Bảng chữ 3 loại: công thức phần tử giữa khớp brute force; '0120' -> 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi bộ ba | brute force | `O(n³)` | `O(1)` | kiểm tra |
| 2. Giữa + tiền tố | `L(j)·R(j)` | `O(n)` | `O(n)` | rõ ràng |
| 3. Giữa + bộ đếm chạy | `seen`, `total − seen` | `O(n)` | `O(1)` | **nộp bài** |
| 4. DP 6 bộ đếm | đếm dãy con `010`, `101` | `O(n)` | `O(1)` | tổng quát hóa |
| 5. `count_pattern` | DP mẫu tổng quát | `O(n·|p|)` | `O(|p|)` | mẫu bất kỳ |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `numberOfWays("0101")`.
2. **Tính tay:** `numberOfWays("0011100")` bằng bảng "phần tử giữa".
3. **Điền TODO:** `count_010(s)` chỉ đếm mẫu `"010"` bằng 3 bộ đếm.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `ways_k(s, k)` — số cách chọn `k` tòa (`i₁ < … < i_k`) sao cho hai tòa liền nhau trong bộ chọn luôn khác loại. Kiểm bằng brute force.

In [37]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_010(s):
    # TODO
    pass

In [38]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_number_of_ways(s):
    # TODO
    pass

In [39]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def ways_k(s, k):
    # TODO: end0[t], end1[t] = số dãy con xen kẽ độ dài t kết thúc bằng '0' / '1'
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Liệt kê `C(4,3) = 4` bộ. (2) `"010"` (0,1,2), `"101"` (1,2,3), `"010"`? (0,1,3) = `"011"` ✗... thử cả 4. (3) Đáp án 2.

**Bài 2.** (1) `total0 = 4`, `total1 = 3`. (2) Các `'1'` ở giữa: mỗi cái có 2 số `0` bên trái, 2 bên phải ⇒ `4` mỗi cái ⇒ `12`. (3) Các `'0'`: chỉ hai số `0` cuối có `'1'` bên trái (3) nhưng bên phải không có `'1'` ⇒ 0. Tổng `12`.

**Bài 3.** (1) `c0, c01, c010`. (2) Gặp `'0'`: `c010 += c01` rồi `c0 += 1`. (3) Gặp `'1'`: `c01 += c0`.

**Bài 4.** (1) `total0`, `total1`. (2) `ch == '0'` ⇒ `seen1 · (total1 − seen1)`. (3) Tăng `seen` sau khi cộng.

**Bài 5.** (1) Gặp `'0'`: `end0[t] += end1[t−1]` với `t` từ `k` xuống 2, và `end0[1] += 1`. (2) Tương tự cho `'1'`. (3) Đáp án `end0[k] + end1[k]`.

## 19. Đáp án tham khảo

In [40]:
# Bài 1, 2
assert Solution().numberOfWays("0101") == ways_brute("0101")
assert Solution().numberOfWays("0011100") == ways_brute("0011100") == 12
print("Bài 1:", Solution().numberOfWays("0101"), "| Bài 2: 12")
trace_running("0011100")

Bài 1: 2 | Bài 2: 12
j=0 '0' giữa của '101': 0 x 3 = 0 -> ans=0
j=1 '0' giữa của '101': 0 x 3 = 0 -> ans=0
j=2 '1' giữa của '010': 2 x 2 = 4 -> ans=4
j=3 '1' giữa của '010': 2 x 2 = 4 -> ans=8
j=4 '1' giữa của '010': 2 x 2 = 4 -> ans=12
j=5 '0' giữa của '101': 3 x 0 = 0 -> ans=12
j=6 '0' giữa của '101': 3 x 0 = 0 -> ans=12


12

In [41]:
# Bài 3, 4
def count_010_ans(s):
    c0 = c01 = c010 = 0
    for ch in s:
        if ch == "0": c010 += c01; c0 += 1
        else: c01 += c0
    return c010

my_number_of_ways_ans = ways_running
rng = random.Random(3)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(3, 14)))
    assert count_010_ans(s) == sum(1 for i, j, k in combinations(range(len(s)), 3) if s[i] + s[j] + s[k] == "010")
    assert my_number_of_ways_ans(s) == ways_brute(s)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [42]:
# Bài 5
def ways_k_ans(s, k):
    end0 = [0] * (k + 1); end1 = [0] * (k + 1)
    for ch in s:
        if ch == "0":
            for t in range(k, 1, -1): end0[t] += end1[t - 1]
            end0[1] += 1
        else:
            for t in range(k, 1, -1): end1[t] += end0[t - 1]
            end1[1] += 1
    return end0[k] + end1[k]

def ways_k_brute(s, k):
    return sum(1 for idx in combinations(range(len(s)), k) if all(s[idx[t]] != s[idx[t + 1]] for t in range(k - 1)))

rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 12))); k = rng.randint(1, 5)
    assert ways_k_ans(s, k) == ways_k_brute(s, k)
assert all(ways_k_ans(s, 3) == e for s, e in examples)
print("Bài 5: OK; '001101' với k = 2, 3, 4 ->", [ways_k_ans("001101", k) for k in (2, 3, 4)])

Bài 5: OK; '001101' với k = 2, 3, 4 -> [9, 6, 4]


**Giải thích thêm cho Bài 5.** `end0[t]` = số dãy con xen kẽ độ dài `t` **kết thúc bằng `'0'`** trong tiền tố đã đọc. Một dãy xen kẽ độ dài `t` kết thúc tại một chữ `'0'` mới là: một dãy xen kẽ độ dài `t − 1` kết thúc bằng `'1'` (trước đó) nối thêm chữ `'0'` này — hoặc chính chữ `'0'` đứng một mình khi `t = 1`. Duyệt `t` giảm dần để không dùng chữ hiện tại hai lần. Bảng dưới in số cách theo `k` cho ví dụ 1 — với `k = 3` ra đúng 6.

In [43]:
s = "001101"
for k in range(1, len(s) + 1):
    print(f"k={k}: {ways_k_ans(s, k):3d} cách | brute: {ways_k_brute(s, k)}")

k=1:   6 cách | brute: 6
k=2:   9 cách | brute: 9
k=3:   6 cách | brute: 6
k=4:   4 cách | brute: 4
k=5:   0 cách | brute: 0
k=6:   0 cách | brute: 0


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "cố định phần tử giữa":**

```python
total0 = s.count('0'); total1 = len(s) - total0
seen0 = seen1 = ans = 0
for ch in s:
    if ch == '0': ans += seen1 * (total1 - seen1); seen0 += 1
    else:         ans += seen0 * (total0 - seen0); seen1 += 1
return ans
```

In [44]:
def template(s):
    total0 = s.count("0"); total1 = len(s) - total0; seen0 = seen1 = ans = 0
    for ch in s:
        if ch == "0": ans += seen1 * (total1 - seen1); seen0 += 1
        else: ans += seen0 * (total0 - seen0); seen1 += 1
    return ans
print([template(s) for s, _ in examples])

[6, 0]


**Checklist:**

- [ ] Đếm **cả** `"010"` và `"101"`.
- [ ] Phần tử giữa nhân số **khác loại** ở hai bên.
- [ ] Số bên phải = tổng − đã thấy (không trừ thừa).
- [ ] DP đếm mẫu: cập nhật bậc **cao trước** thấp.
- [ ] Đáp án lớn (`long`).
- [ ] Đã thử `n = 3`, chuỗi toàn một ký tự, xen kẽ.